In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS olist")
spark.sql("USE CATALOG olist")
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("USE DATABASE bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS olist.default.landing")

In [0]:
LANDING_PATH = "/Volumes/olist/default/landing"

In [0]:
from pyspark.sql import functions as F

def ingest_csv(file_name: str, table_name: str) -> None:
    """
    Lê um arquivo CSV da Landing Zone e salva como Delta Table na camada Bronze.
    Adiciona coluna timestamp_ingestion conforme requisito da atividade.
    
    Args:
        file_name: Nome do arquivo CSV (ex: 'olist_customers_dataset.csv')
        table_name: Nome da tabela de destino na Bronze (ex: 'tb_customers')
    """
    file_path = f"{LANDING_PATH}/{file_name}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(file_path)
    )

    df = df.withColumn("timestamp_ingestion", F.current_timestamp())

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"olist.bronze.{table_name}")
    )

    print(f"✅ {file_name} -> olist.bronze.{table_name} ({df.count()} registros)")

In [0]:
# Mapeamento de arquivos para tabelas 
arquivos = [
    ("olist_customers_dataset.csv",            "tb_customers"),
    ("olist_geolocation_dataset.csv",          "tb_geolocalizacao"),
    ("olist_order_items_dataset.csv",          "tb_order_items"),
    ("olist_order_payments_dataset.csv",       "tb_order_payments"),
    ("olist_order_reviews_dataset.csv",        "tb_order_reviews"),
    ("olist_orders_dataset.csv",               "tb_orders"),
    ("olist_products_dataset.csv",             "tb_products"),
    ("olist_sellers_dataset.csv",              "tb_sellers"),
    ("product_category_name_translation.csv",  "tb_product_category_name_translation"),
]

for file_name, table_name in arquivos:
    ingest_csv(file_name, table_name)

In [0]:
import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

data_inicio = "01-01-2016"
data_fim = "12-31-2018"

In [0]:
def buscar_cotacao_dolar(data_inicio: str, data_fim: str) -> list:
    """
    Extrai a cotação do dólar via API do Banco Central do Brasil (PTAX).
    
    Args:
        data_inicio: Data inicial no formato MM-DD-AAAA
        data_fim: Data final no formato MM-DD-AAAA
    
    Returns:
        Lista de dicionários com dataHoraCotacao e cotacaoCompra
    """
    url = (
        f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
        f"?@dataInicial='{data_inicio}'"
        f"&@dataFinalCotacao='{data_fim}'"
        f"&$select=dataHoraCotacao,cotacaoCompra"
        f"&$format=json"
    )

    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Erro ao chamar a API: {response.status_code} - {response.text}")

    dados = response.json().get("value", [])
    print(f"✅ {len(dados)} registros retornados pela API")
    return dados

In [0]:
dados_cotacao = buscar_cotacao_dolar(data_inicio, data_fim)

# Define schema explícito para garantir tipagem correta
schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra",   DoubleType(), True),
])

# Cria o DataFrame com os dados retornados pela API
df_cotacao = spark.createDataFrame(dados_cotacao, schema=schema)

# Adiciona coluna de timestamp de ingestão
df_cotacao = df_cotacao.withColumn("timestamp_ingestion", F.current_timestamp())

(
    df_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.bronze.tb_cotacao_dolar")
)

print(f"✅ Cotação do dólar -> olist.bronze.tb_cotacao_dolar ({df_cotacao.count()} registros)")